In [0]:
# Configuración de credenciales para acceso al Blob Storage

spark.conf.set(
    "fs.azure.account.key.diplomaturastorage2026.blob.core.windows.net",
    "TU_KEY_AQUI"
)

In [0]:
# Definición de rutas RAW y BRONZE

ruta_raw = "wasbs://raw@diplomaturastorage2026.blob.core.windows.net/"
ruta_bronze = "wasbs://bronze@diplomaturastorage2026.blob.core.windows.net/"

In [0]:
# Configuración de conexión a Azure SQL Database

jdbc_hostname = "diplomatura-server.database.windows.net"
jdbc_port = 1433
jdbc_database = "diplomatura-db"

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database}"

connection_properties = {
    "user": "TU_USUARIO",
    "password": "TU_KEY_AQUI",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
# Lectura de datasets desde RAW

df_ventas_raw = spark.read \
    .option("header", "true") \
    .csv(ruta_raw + "ventas")

df_productos_raw = spark.read \
    .option("header", "true") \
    .csv(ruta_raw + "productos")

df_clientes_raw = spark.read \
    .option("header", "true") \
    .csv(ruta_raw + "clientes")

In [0]:
# Lectura de tabla tiendas desde Azure SQL Database

df_tiendas_sql = spark.read.jdbc(
    url=jdbc_url,
    table="tiendas",
    properties=connection_properties
)

In [0]:
# Validación de cantidad de registros cargados

print("Ventas RAW:", df_ventas_raw.count())
print("Productos RAW:", df_productos_raw.count())
print("Clientes RAW:", df_clientes_raw.count())
print("Tiendas SQL:", df_tiendas_sql.count())

Ventas RAW: 2003
Productos RAW: 10
Clientes RAW: 100
Tiendas SQL: 5


In [0]:
# Escritura de datasets en formato Delta Bronze

df_ventas_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_bronze + "ventas")

df_productos_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_bronze + "productos")

df_clientes_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_bronze + "clientes")

df_tiendas_sql.write \
    .format("delta") \
    .mode("overwrite") \
    .save(ruta_bronze + "tiendas")

In [0]:
# Verificación de archivos almacenados en Bronze

dbutils.fs.ls(ruta_bronze)

[FileInfo(path='wasbs://bronze@diplomaturastorage2026.blob.core.windows.net/clientes/', name='clientes/', size=0, modificationTime=1778525086000),
 FileInfo(path='wasbs://bronze@diplomaturastorage2026.blob.core.windows.net/productos/', name='productos/', size=0, modificationTime=1778525085000),
 FileInfo(path='wasbs://bronze@diplomaturastorage2026.blob.core.windows.net/tiendas/', name='tiendas/', size=0, modificationTime=1778525088000),
 FileInfo(path='wasbs://bronze@diplomaturastorage2026.blob.core.windows.net/ventas/', name='ventas/', size=0, modificationTime=1778525081000)]

In [0]:
# Lectura y validación final de tablas Bronze

df_ventas_bronze = spark.read \
    .format("delta") \
    .load(ruta_bronze + "ventas")

df_productos_bronze = spark.read \
    .format("delta") \
    .load(ruta_bronze + "productos")

df_clientes_bronze = spark.read \
    .format("delta") \
    .load(ruta_bronze + "clientes")

df_tiendas_bronze = spark.read \
    .format("delta") \
    .load(ruta_bronze + "tiendas")

print("Ventas Bronze:", df_ventas_bronze.count())
print("Productos Bronze:", df_productos_bronze.count())
print("Clientes Bronze:", df_clientes_bronze.count())
print("Tiendas Bronze:", df_tiendas_bronze.count())

Ventas Bronze: 2003
Productos Bronze: 10
Clientes Bronze: 100
Tiendas Bronze: 5
